<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/05_GES_Aware_Genomic_RAG_Cell_7B0_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES-RAG Experiment 2 — Cell 7B0


In [1]:

from collections import OrderedDict
from pathlib import Path
import hashlib
import json
import os
import re
import time

import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE AND PROJECT PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"
NOTEBOOK_NAME = "05_GES_Aware_Genomic_RAG_Cell_7B0_V2.ipynb"
CELL_ID = "7B0"
PACKAGE_VERSION = "v1"

STAGE7_CONFIG_DIR = ROOT / "configs" / "stage7_rag"
STAGE7_DATA_DIR = ROOT / "data_processed" / "stage7_rag"
STAGE7_TABLE_DIR = ROOT / "outputs" / "tables" / "stage7_rag"
STAGE7_QC_DIR = ROOT / "outputs" / "quality_checks" / "stage7_rag"

for directory in [STAGE7_CONFIG_DIR, STAGE7_DATA_DIR, STAGE7_TABLE_DIR, STAGE7_QC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 2. EXACT CELL 7A3 FROZEN INPUTS
# --------------------------------------------------------------------------------------------------

CELL_7A3_SCORE_TABLE = (
    STAGE7_DATA_DIR / "cell_7a3_t1_frozen_ges_and_metadata_scores_v1.parquet"
)
CELL_7A3_SCHEMA = (
    STAGE7_TABLE_DIR / "cell_7a3_t1_score_schema_inventory_v1.csv"
)
CELL_7A3_RANGE_QC = (
    STAGE7_TABLE_DIR / "cell_7a3_t1_score_range_qc_inventory_v1.csv"
)
CELL_7A3_REPORT = (
    STAGE7_QC_DIR / "cell_7a3_t1_score_materialization_report_v1.json"
)
CELL_7A3_QC = (
    STAGE7_QC_DIR / "cell_7a3_t1_score_materialization_qc_v1.json"
)
CELL_7A3_MANIFEST = (
    STAGE7_CONFIG_DIR / "cell_7a3_t1_score_materialization_manifest_v1.json"
)

EXPECTED_CELL_7A3_HASHES = OrderedDict([
    ("score_table", "e9b162c9add5aed34a4d68d8bf625251293a18c549945468e627a498843eb802"),
    ("schema_inventory", "c03933e37192ea538b1d835795d69a031fd07c6eaf7e08b86c47f338925c066a"),
    ("range_qc_inventory", "ed358ea18c2f4c890f6d18b647a8188873a13fd0ccaf5f03470a9778351f8f77"),
    ("materialization_report", "2cc47b0464e115ff5aff6ac30c685e1fdec53183016fb312ed4d3c416d1b819d"),
    ("qc_record", "12532b23cbeb16b5dbeb8c45d53589b0e79a5ebb74315a87de62b4fa5c31e79c"),
    ("manifest", "99bcff934f5e0c15d8357450fb3992eccfeb972ab61ddba8431a843c13b532dd"),
])

CELL_7A3_PATHS = OrderedDict([
    ("score_table", CELL_7A3_SCORE_TABLE),
    ("schema_inventory", CELL_7A3_SCHEMA),
    ("range_qc_inventory", CELL_7A3_RANGE_QC),
    ("materialization_report", CELL_7A3_REPORT),
    ("qc_record", CELL_7A3_QC),
    ("manifest", CELL_7A3_MANIFEST),
])

EXPECTED_CELL_7A3_DECISION = (
    "PASS_STAGE7A3_FROZEN_T1_FULL_GES_NO_STAR_GES_AND_COMBINED_METADATA_SCORES_"
    "MATERIALIZED_CHECKSUM_PROTECTED_T0_PREDICT_PROBA_REPRODUCED_NO_FITTING_"
    "NO_THRESHOLDING_NO_RANKING_NO_RAG_CORPUS_EMBEDDINGS_OR_LLM_NEXT_STAGE_"
    "REQUIRES_SEPARATE_PROTOCOL_FREEZE"
)

EXPECTED_ROWS = 100_920
EXPECTED_COLUMNS = 28

REQUIRED_SCORE_COLUMNS = {
    "rcv_accession",
    "target_gene",
    "classification_axis",
    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "recency_missing_instability_component",
    "submitter_instability_risk",
    "entropy_instability_risk",
    "combined_metadata_instability_risk",
    "full_ges_p_stable_t1",
    "full_ges_instability_risk_t1",
    "no_star_ges_p_stable_t1",
    "no_star_ges_instability_risk_t1",
}


# --------------------------------------------------------------------------------------------------
# 3. OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

OUTPUTS = OrderedDict([
    (
        "protocol",
        STAGE7_CONFIG_DIR / "cell_7b0_downstream_rag_protocol_v1.json",
    ),
    (
        "condition_inventory",
        STAGE7_TABLE_DIR / "cell_7b0_experimental_condition_inventory_v1.csv",
    ),
    (
        "metric_inventory",
        STAGE7_TABLE_DIR / "cell_7b0_evaluation_metric_inventory_v1.csv",
    ),
    (
        "control_inventory",
        STAGE7_TABLE_DIR / "cell_7b0_leakage_and_invariance_control_inventory_v1.csv",
    ),
    (
        "qc",
        STAGE7_QC_DIR / "cell_7b0_downstream_rag_protocol_qc_v1.json",
    ),
    (
        "manifest",
        STAGE7_CONFIG_DIR / "cell_7b0_downstream_rag_protocol_manifest_v1.json",
    ),
])


# --------------------------------------------------------------------------------------------------
# 4. HASHING, SIDECARS, AND DETERMINISTIC WRITERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return Path(str(path) + ".sha256")


def read_sidecar_hash(path: Path) -> str:
    text = Path(path).read_text(encoding="utf-8").strip()
    values = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not values:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return values[0].lower()


def sidecar_is_valid(path: Path) -> bool:
    path = Path(path)
    sidecar = sidecar_path(path)
    return (
        path.exists()
        and sidecar.exists()
        and read_sidecar_hash(sidecar) == sha256_file(path)
    )


def verify_exact_hash(label: str, path: Path, expected_hash: str) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing frozen artifact for {label}: {path}")
    observed_hash = sha256_file(path)
    if observed_hash != expected_hash:
        raise AssertionError(
            f"SHA-256 mismatch for {label}.\n"
            f"Expected: {expected_hash}\n"
            f"Observed: {observed_hash}\n"
            f"Path: {path}"
        )
    return observed_hash


def json_native(value):
    if isinstance(value, dict):
        return {str(key): json_native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_native(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    proposed_hash = sha256_file(temporary)

    if path.exists():
        existing_hash = sha256_file(path)
        if existing_hash != proposed_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(
                "Refusing to overwrite a nonidentical frozen Cell 7B0 artifact.\n"
                f"Path: {path}\n"
                f"Existing SHA-256: {existing_hash}\n"
                f"Proposed SHA-256: {proposed_hash}"
            )
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)

    return sha256_file(path)


def stable_write_json(path: Path, payload: dict) -> str:
    data = (
        json.dumps(
            json_native(payload),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, data)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    data = frame.to_csv(index=False, lineterminator="\n").encode("utf-8")
    return stable_write_bytes(path, data)


def write_sidecar(path: Path) -> str:
    path = Path(path)
    payload = f"{sha256_file(path)}  {path.name}\n".encode("utf-8")
    stable_write_bytes(sidecar_path(path), payload)
    return sha256_file(sidecar_path(path))


def normalize_qc_count(value, field_name: str) -> int:
    """
    Normalize QC summary fields that may be stored either as integer counts
    or as explicit lists/dictionaries of checks.
    """
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        return int(value)
    if isinstance(value, (list, tuple, set, dict)):
        return len(value)
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.isdigit():
            return int(stripped)
    if value is None:
        return 0
    raise TypeError(
        f"Unsupported QC field type for {field_name}: "
        f"{type(value).__name__} ({value!r})"
    )


def qc_summary_counts(payload: dict) -> tuple[int, int, int]:
    passed = normalize_qc_count(payload.get("passed_checks", 0), "passed_checks")
    failed = normalize_qc_count(payload.get("failed_checks", 0), "failed_checks")

    total_raw = payload.get("total_checks")
    if total_raw is None:
        checks = payload.get("checks")
        total = len(checks) if isinstance(checks, (list, tuple, dict)) else passed + failed
    else:
        total = normalize_qc_count(total_raw, "total_checks")

    return passed, failed, total


# --------------------------------------------------------------------------------------------------
# 5. REVERIFY CELL 7A3 AND READ ONLY PARQUET METADATA
# --------------------------------------------------------------------------------------------------

observed_upstream_hashes = OrderedDict()
for label, path in CELL_7A3_PATHS.items():
    observed_upstream_hashes[label] = verify_exact_hash(
        label,
        path,
        EXPECTED_CELL_7A3_HASHES[label],
    )
    if not sidecar_is_valid(path):
        raise AssertionError(f"Invalid or missing SHA-256 sidecar for {label}: {path}")

cell_7a3_manifest = json.loads(CELL_7A3_MANIFEST.read_text(encoding="utf-8"))
cell_7a3_qc = json.loads(CELL_7A3_QC.read_text(encoding="utf-8"))

manifest_qc_passed, manifest_qc_failed, manifest_qc_total = qc_summary_counts(
    cell_7a3_manifest.get("qc", {})
)
record_qc_passed, record_qc_failed, record_qc_total = qc_summary_counts(cell_7a3_qc)

parquet_file = pq.ParquetFile(CELL_7A3_SCORE_TABLE)
observed_rows = int(parquet_file.metadata.num_rows)
observed_columns = len(parquet_file.schema_arrow.names)
observed_schema = set(parquet_file.schema_arrow.names)


# --------------------------------------------------------------------------------------------------
# 6. FREEZE THE DOWNSTREAM RAG PROTOCOL
# --------------------------------------------------------------------------------------------------

conditions = pd.DataFrame([
    {
        "condition_id": "A",
        "condition_name": "Standard semantic-only RAG",
        "role": "baseline",
        "candidate_pool": "same fixed semantic top-20",
        "final_context": "top-5 by semantic rank",
        "quality_signal": "none",
        "quality_action": "none",
        "score_exposed_to_llm": False,
        "hard_exclusion": False,
    },
    {
        "condition_id": "B",
        "condition_name": "Review/conflict-aware RAG",
        "role": "mandatory metadata baseline",
        "candidate_pool": "same fixed semantic top-20",
        "final_context": "top-5 after fixed soft rank fusion",
        "quality_signal": "1 - mean(review_stars_instability_risk, conflict_instability_risk)",
        "quality_action": "soft reranking only",
        "score_exposed_to_llm": False,
        "hard_exclusion": False,
    },
    {
        "condition_id": "C",
        "condition_name": "Combined-metadata-aware RAG",
        "role": "mandatory strongest metadata comparator",
        "candidate_pool": "same fixed semantic top-20",
        "final_context": "top-5 after fixed soft rank fusion",
        "quality_signal": "1 - combined_metadata_instability_risk",
        "quality_action": "soft reranking only",
        "score_exposed_to_llm": False,
        "hard_exclusion": False,
    },
    {
        "condition_id": "D",
        "condition_name": "Full-GES-aware RAG",
        "role": "primary intervention",
        "candidate_pool": "same fixed semantic top-20",
        "final_context": "top-5 after fixed soft rank fusion",
        "quality_signal": "full_ges_p_stable_t1",
        "quality_action": "soft reranking only",
        "score_exposed_to_llm": False,
        "hard_exclusion": False,
    },
    {
        "condition_id": "E",
        "condition_name": "No-star-GES-aware RAG",
        "role": "feature-ablation comparator",
        "candidate_pool": "same fixed semantic top-20",
        "final_context": "top-5 after fixed soft rank fusion",
        "quality_signal": "no_star_ges_p_stable_t1",
        "quality_action": "soft reranking only",
        "score_exposed_to_llm": False,
        "hard_exclusion": False,
    },
    {
        "condition_id": "F",
        "condition_name": "Random-quality control RAG",
        "role": "negative control",
        "candidate_pool": "same fixed semantic top-20",
        "final_context": "top-5 after fixed soft rank fusion",
        "quality_signal": "deterministic within-pool random quality rank; seed 20260722",
        "quality_action": "soft reranking only",
        "score_exposed_to_llm": False,
        "hard_exclusion": False,
    },
])

metrics = pd.DataFrame([
    {
        "metric_id": "M1",
        "metric_name": "citation_supported_factual_accuracy",
        "role": "primary",
        "direction": "higher_is_better",
        "unit": "question-level proportion of atomic factual claims that are correct and supported by cited context",
        "primary_comparison": "D_vs_A",
    },
    {
        "metric_id": "M2",
        "metric_name": "answer_correctness",
        "role": "secondary",
        "direction": "higher_is_better",
        "unit": "frozen rubric score",
        "primary_comparison": "",
    },
    {
        "metric_id": "M3",
        "metric_name": "citation_precision",
        "role": "secondary",
        "direction": "higher_is_better",
        "unit": "supported citations / cited sources",
        "primary_comparison": "",
    },
    {
        "metric_id": "M4",
        "metric_name": "unsupported_claim_rate",
        "role": "secondary safety",
        "direction": "lower_is_better",
        "unit": "unsupported atomic claims / all atomic factual claims",
        "primary_comparison": "",
    },
    {
        "metric_id": "M5",
        "metric_name": "conflict_recognition_accuracy",
        "role": "secondary",
        "direction": "higher_is_better",
        "unit": "binary or rubric-based recognition of conflicting evidence",
        "primary_comparison": "",
    },
    {
        "metric_id": "M6",
        "metric_name": "appropriate_abstention",
        "role": "secondary safety",
        "direction": "higher_is_better",
        "unit": "correct abstention or qualified answer under insufficient/conflicting evidence",
        "primary_comparison": "",
    },
    {
        "metric_id": "M7",
        "metric_name": "confidence_brier_score",
        "role": "secondary calibration",
        "direction": "lower_is_better",
        "unit": "Brier score from frozen answer confidence and correctness",
        "primary_comparison": "",
    },
    {
        "metric_id": "M8",
        "metric_name": "unsafe_or_overconfident_recommendation_rate",
        "role": "secondary safety",
        "direction": "lower_is_better",
        "unit": "proportion of outputs containing unsupported patient-level or definitive clinical recommendations",
        "primary_comparison": "",
    },
])

controls = pd.DataFrame([
    {
        "control_id": "C01",
        "control": "Question construction is score-blind",
        "requirement": "No Cell 7A3 score column may be loaded until the question set and answer-key specification are frozen.",
    },
    {
        "control_id": "C02",
        "control": "Same question set",
        "requirement": "Every condition receives the identical frozen questions.",
    },
    {
        "control_id": "C03",
        "control": "Same semantic candidate pool",
        "requirement": "All six conditions use the same semantic top-20 candidates for each question.",
    },
    {
        "control_id": "C04",
        "control": "Same context size",
        "requirement": "All six conditions provide exactly five evidence units unless fewer than five eligible units exist.",
    },
    {
        "control_id": "C05",
        "control": "Same evidence representation",
        "requirement": "Evidence packet text and field ordering are identical across conditions.",
    },
    {
        "control_id": "C06",
        "control": "Same embedding model",
        "requirement": "One exact embedding model/version and preprocessing policy must be frozen before embedding generation.",
    },
    {
        "control_id": "C07",
        "control": "Same LLM and generation settings",
        "requirement": "One exact model/version, prompt, temperature, seed where supported, and token budget must be frozen before calls.",
    },
    {
        "control_id": "C08",
        "control": "Scores hidden from the LLM",
        "requirement": "Primary-condition score values are used only for reranking and are not printed into the context or prompt.",
    },
    {
        "control_id": "C09",
        "control": "No hard evidence exclusion",
        "requirement": "No evidence is removed using a GES or metadata threshold in the primary experiment.",
    },
    {
        "control_id": "C10",
        "control": "No threshold or weight tuning",
        "requirement": "Rank-fusion constants, top-k values, metrics, and comparisons cannot be optimized using answer outcomes.",
    },
    {
        "control_id": "C11",
        "control": "Blinded condition labels",
        "requirement": "Evaluation records use randomized condition aliases so adjudicators do not know the intervention arm.",
    },
    {
        "control_id": "C12",
        "control": "Paired analysis",
        "requirement": "Condition differences are calculated within the same question and candidate pool.",
    },
    {
        "control_id": "C13",
        "control": "EGFR remains exploratory",
        "requirement": "Primary germline analyses are reported separately from the EGFR somatic-oncology case study.",
    },
    {
        "control_id": "C14",
        "control": "No patient-level decision support",
        "requirement": "Questions and outputs are research evidence-synthesis tasks, not clinical recommendations.",
    },
])

protocol = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "protocol_name": "GES-aware genomic RAG controlled interventional pilot",
    "upstream_cell_7a3_manifest_sha256": EXPECTED_CELL_7A3_HASHES["manifest"],
    "upstream_cell_7a3_score_table_sha256": EXPECTED_CELL_7A3_HASHES["score_table"],
    "scientific_status": {
        "experiment_1_interpretation": (
            "GES is a weak relative warning/prioritization signal; it is not a calibrated "
            "clinical probability or validated clinical threshold."
        ),
        "experiment_2_scope": "controlled research pilot only",
        "clinical_use_authorized": False,
        "patient_level_recommendations_authorized": False,
    },
    "research_question": (
        "Does fixed soft reranking of a common semantic candidate pool using frozen evidence-stability "
        "signals improve citation-supported factual accuracy, evidence fidelity, conflict recognition, "
        "uncertainty communication, and appropriate abstention in genomic question answering?"
    ),
    "primary_hypothesis": (
        "Condition D, Full-GES-aware soft reranking, improves the primary endpoint relative to "
        "Condition A, semantic-only RAG, on the same frozen questions and candidate pools."
    ),
    "mandatory_comparator_hypotheses": [
        "Compare Full-GES-aware RAG with review/conflict-aware RAG.",
        "Compare Full-GES-aware RAG with combined-metadata-aware RAG.",
        "Compare Full-GES-aware RAG with no-star-GES-aware RAG.",
        "Verify that deterministic random-quality reranking does not reproduce a meaningful benefit.",
    ],
    "evidence_unit": "one T1 RCV-level variant-condition aggregate with nested-SCV provenance summary",
    "question_set": {
        "target_question_count": 80,
        "genes": {
            "BRCA1": 20,
            "BRCA2": 20,
            "MLH1": 20,
            "EGFR": 20,
        },
        "question_types_per_gene": {
            "aggregate_interpretation_summary": 5,
            "conflict_recognition": 5,
            "evidence_rigor_and_provenance": 5,
            "uncertainty_or_abstention": 5,
        },
        "construction": "score-blind and frozen before score-aware context assembly",
        "answer_key_and_rubric": "must be frozen before any LLM call",
        "egfr_role": "exploratory and separately reported",
    },
    "retrieval_and_reranking": {
        "semantic_candidate_pool_k": 20,
        "final_context_k": 5,
        "candidate_pool_invariant_across_conditions": True,
        "soft_rank_fusion": {
            "method": "weighted reciprocal rank fusion",
            "formula": "0.75/(60 + semantic_rank) + 0.25/(60 + quality_rank)",
            "semantic_weight": 0.75,
            "quality_weight": 0.25,
            "rrf_constant": 60,
            "tie_breaker": "rcv_accession ascending",
        },
        "primary_intervention": "soft reranking only",
        "score_values_exposed_to_llm": False,
        "hard_exclusion_authorized": False,
        "warning_label_sensitivity_authorized": False,
    },
    "experimental_conditions": conditions.to_dict("records"),
    "evaluation": {
        "primary_endpoint": "citation_supported_factual_accuracy",
        "primary_comparison": "D_vs_A",
        "key_secondary_comparisons": ["D_vs_B", "D_vs_C", "D_vs_E", "F_vs_A"],
        "paired_unit": "question",
        "bootstrap_replicates": 2000,
        "bootstrap_unit": "question",
        "confidence_interval": "paired percentile 95% CI",
        "multiplicity": (
            "Primary inference is D_vs_A only. Key secondary comparisons are reported with Holm "
            "adjustment within the primary endpoint family."
        ),
        "evaluation_rubric_status": "must be separately frozen before LLM execution",
        "condition_blinding_required": True,
    },
    "generation_invariants_to_freeze_before_execution": [
        "exact embedding model and version",
        "exact text normalization and chunking policy",
        "exact LLM model and dated version where available",
        "exact system and user prompts",
        "temperature and sampling settings",
        "token budget and response schema",
        "condition-alias randomization map",
        "answer key, scoring rubric, and adjudication procedure",
    ],
    "prohibited_in_cell_7b0": [
        "evidence-packet materialization",
        "RAG corpus construction",
        "embedding generation",
        "semantic retrieval",
        "quality reranking",
        "question generation or selection",
        "prompt generation",
        "LLM calls",
        "model fitting or recalibration",
        "threshold optimization",
        "hard evidence exclusion",
    ],
    "next_authorized_cell": {
        "cell_id": "7B1",
        "scope": (
            "Score-blind evidence-unit, eligibility, field-derivability, and question-stratum "
            "preflight only."
        ),
        "may_load_cell_7a3_scores": False,
        "may_materialize_evidence_packets": False,
        "may_construct_corpus": False,
        "may_generate_embeddings": False,
        "may_run_retrieval": False,
        "may_generate_questions": False,
        "may_generate_prompts": False,
        "may_call_llm": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 7. FAIL-BEFORE-OUTPUT QC
# --------------------------------------------------------------------------------------------------

condition_ids = conditions["condition_id"].tolist()
condition_names = conditions["condition_name"].tolist()
metric_names = set(metrics["metric_name"])
control_ids = controls["control_id"].tolist()

prewrite_checks = OrderedDict([
    ("cell_7a3_manifest_exact_hash", observed_upstream_hashes["manifest"] == EXPECTED_CELL_7A3_HASHES["manifest"]),
    ("cell_7a3_score_table_exact_hash", observed_upstream_hashes["score_table"] == EXPECTED_CELL_7A3_HASHES["score_table"]),
    ("all_six_cell_7a3_artifacts_verified", len(observed_upstream_hashes) == 6),
    ("all_six_cell_7a3_sidecars_valid", all(sidecar_is_valid(path) for path in CELL_7A3_PATHS.values())),
    ("cell_7a3_terminal_decision_exact", cell_7a3_manifest.get("terminal_decision") == EXPECTED_CELL_7A3_DECISION),
    ("cell_7a3_no_later_cell_auto_authorized", cell_7a3_manifest.get("next_authorized_cell") is None),
    ("cell_7a3_qc_50_of_50", (
        manifest_qc_passed == 50
        and manifest_qc_failed == 0
        and manifest_qc_total == 50
    )),
    ("cell_7a3_qc_payload_pass", (
        record_qc_passed == 50
        and record_qc_failed == 0
        and record_qc_total == 50
    )),
    ("score_table_row_count", observed_rows == EXPECTED_ROWS),
    ("score_table_column_count", observed_columns == EXPECTED_COLUMNS),
    ("required_score_columns_present", REQUIRED_SCORE_COLUMNS.issubset(observed_schema)),
    ("six_conditions_defined", len(conditions) == 6),
    ("condition_ids_exact", condition_ids == ["A", "B", "C", "D", "E", "F"]),
    ("condition_ids_unique", len(condition_ids) == len(set(condition_ids))),
    ("condition_names_unique", len(condition_names) == len(set(condition_names))),
    ("semantic_baseline_present", "Standard semantic-only RAG" in condition_names),
    ("review_conflict_baseline_present", "Review/conflict-aware RAG" in condition_names),
    ("combined_metadata_baseline_present", "Combined-metadata-aware RAG" in condition_names),
    ("full_ges_intervention_present", "Full-GES-aware RAG" in condition_names),
    ("no_star_comparator_present", "No-star-GES-aware RAG" in condition_names),
    ("random_control_present", "Random-quality control RAG" in condition_names),
    ("common_candidate_pool_frozen", conditions["candidate_pool"].nunique() == 1),
    ("context_size_frozen", protocol["retrieval_and_reranking"]["final_context_k"] == 5),
    ("candidate_pool_size_frozen", protocol["retrieval_and_reranking"]["semantic_candidate_pool_k"] == 20),
    ("semantic_weight_frozen", protocol["retrieval_and_reranking"]["soft_rank_fusion"]["semantic_weight"] == 0.75),
    ("quality_weight_frozen", protocol["retrieval_and_reranking"]["soft_rank_fusion"]["quality_weight"] == 0.25),
    ("rrf_constant_frozen", protocol["retrieval_and_reranking"]["soft_rank_fusion"]["rrf_constant"] == 60),
    ("all_scores_hidden_from_llm", conditions["score_exposed_to_llm"].eq(False).all()),
    ("no_hard_exclusion", conditions["hard_exclusion"].eq(False).all()),
    ("primary_endpoint_present", "citation_supported_factual_accuracy" in metric_names),
    ("primary_comparison_exact", protocol["evaluation"]["primary_comparison"] == "D_vs_A"),
    ("bootstrap_replicates_frozen", protocol["evaluation"]["bootstrap_replicates"] == 2000),
    ("question_count_frozen", protocol["question_set"]["target_question_count"] == 80),
    ("question_gene_counts_sum", sum(protocol["question_set"]["genes"].values()) == 80),
    ("question_type_counts_per_gene_sum", sum(protocol["question_set"]["question_types_per_gene"].values()) == 20),
    ("egfr_exploratory", protocol["question_set"]["egfr_role"] == "exploratory and separately reported"),
    ("score_blind_question_construction", protocol["question_set"]["construction"].startswith("score-blind")),
    ("clinical_use_false", protocol["scientific_status"]["clinical_use_authorized"] is False),
    ("patient_recommendations_false", protocol["scientific_status"]["patient_level_recommendations_authorized"] is False),
    ("cell_7b1_only_authorized", protocol["next_authorized_cell"]["cell_id"] == "7B1"),
    ("cell_7b1_cannot_load_scores", protocol["next_authorized_cell"]["may_load_cell_7a3_scores"] is False),
    ("cell_7b1_no_packet_materialization", protocol["next_authorized_cell"]["may_materialize_evidence_packets"] is False),
    ("cell_7b1_no_corpus", protocol["next_authorized_cell"]["may_construct_corpus"] is False),
    ("cell_7b1_no_embeddings", protocol["next_authorized_cell"]["may_generate_embeddings"] is False),
    ("cell_7b1_no_retrieval", protocol["next_authorized_cell"]["may_run_retrieval"] is False),
    ("cell_7b1_no_questions", protocol["next_authorized_cell"]["may_generate_questions"] is False),
    ("cell_7b1_no_prompts", protocol["next_authorized_cell"]["may_generate_prompts"] is False),
    ("cell_7b1_no_llm", protocol["next_authorized_cell"]["may_call_llm"] is False),
    ("control_ids_unique", len(control_ids) == len(set(control_ids))),
    ("fourteen_controls_frozen", len(controls) == 14),
])

failed_prewrite = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed_prewrite:
    raise RuntimeError(
        "Cell 7B0 failed before output. Failed checks:\n- "
        + "\n- ".join(failed_prewrite)
    )


# --------------------------------------------------------------------------------------------------
# 8. WRITE, CHECKSUM, READ BACK, AND FREEZE
# --------------------------------------------------------------------------------------------------

stable_write_json(OUTPUTS["protocol"], protocol)
stable_write_csv(OUTPUTS["condition_inventory"], conditions)
stable_write_csv(OUTPUTS["metric_inventory"], metrics)
stable_write_csv(OUTPUTS["control_inventory"], controls)

for key in ["protocol", "condition_inventory", "metric_inventory", "control_inventory"]:
    write_sidecar(OUTPUTS[key])

qc_payload = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "upstream_cell_7a3_manifest_sha256": EXPECTED_CELL_7A3_HASHES["manifest"],
    "upstream_cell_7a3_score_table_sha256": EXPECTED_CELL_7A3_HASHES["score_table"],
    "passed_checks": len(prewrite_checks),
    "failed_checks": 0,
    "total_checks": len(prewrite_checks),
    "checks": [
        {"check": name, "passed": bool(passed)}
        for name, passed in prewrite_checks.items()
    ],
    "scientific_operations": {
        "evidence_packets_materialized": False,
        "rag_corpus_constructed": False,
        "embeddings_generated": False,
        "retrieval_executed": False,
        "reranking_executed": False,
        "questions_generated": False,
        "prompts_generated": False,
        "llm_called": False,
        "model_fitted_or_recalibrated": False,
        "threshold_or_weight_optimized": False,
        "hard_exclusion_applied": False,
    },
    "decision": (
        "PASS_STAGE7B0_DOWNSTREAM_RAG_PROTOCOL_FROZEN_CHECKSUM_PROTECTED_"
        "STAGE7A3_REVERIFIED_PRIMARY_SOFT_RERANKING_AND_MANDATORY_COMPARATOR_"
        "ARMS_PRESPECIFIED_NO_EVIDENCE_PACKETS_CORPUS_EMBEDDINGS_RETRIEVAL_"
        "QUESTIONS_PROMPTS_OR_LLM_CELL7B1_SCORE_BLIND_EVIDENCE_AND_QUESTION_"
        "PREFLIGHT_ONLY_AUTHORIZED"
    ),
}
stable_write_json(OUTPUTS["qc"], qc_payload)
write_sidecar(OUTPUTS["qc"])

output_records = []
for key in ["protocol", "condition_inventory", "metric_inventory", "control_inventory", "qc"]:
    path = OUTPUTS[key]
    if not sidecar_is_valid(path):
        raise AssertionError(f"Output sidecar verification failed for {key}: {path}")
    output_records.append({
        "artifact": key,
        "path": str(path),
        "sha256": sha256_file(path),
        "sidecar_path": str(sidecar_path(path)),
        "sidecar_sha256": sha256_file(sidecar_path(path)),
    })

terminal_decision = qc_payload["decision"]

manifest_payload = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "upstream_artifacts": [
        {
            "artifact": label,
            "path": str(CELL_7A3_PATHS[label]),
            "sha256": observed_upstream_hashes[label],
        }
        for label in CELL_7A3_PATHS
    ],
    "output_artifacts": output_records,
    "qc": {
        "path": str(OUTPUTS["qc"]),
        "sha256": sha256_file(OUTPUTS["qc"]),
        "passed_checks": len(prewrite_checks),
        "failed_checks": 0,
        "total_checks": len(prewrite_checks),
    },
    "scientific_boundary": qc_payload["scientific_operations"],
    "terminal_decision": terminal_decision,
    "next_authorized_cell": protocol["next_authorized_cell"],
}
stable_write_json(OUTPUTS["manifest"], manifest_payload)
write_sidecar(OUTPUTS["manifest"])

if not sidecar_is_valid(OUTPUTS["manifest"]):
    raise AssertionError("Cell 7B0 manifest sidecar verification failed.")

protocol_readback = json.loads(OUTPUTS["protocol"].read_text(encoding="utf-8"))
qc_readback = json.loads(OUTPUTS["qc"].read_text(encoding="utf-8"))
manifest_readback = json.loads(OUTPUTS["manifest"].read_text(encoding="utf-8"))

readback_checks = OrderedDict([
    ("protocol_readback_primary_comparison", protocol_readback["evaluation"]["primary_comparison"] == "D_vs_A"),
    ("protocol_readback_six_conditions", len(protocol_readback["experimental_conditions"]) == 6),
    ("protocol_readback_no_hard_exclusion", protocol_readback["retrieval_and_reranking"]["hard_exclusion_authorized"] is False),
    ("qc_readback_all_passed", qc_readback["failed_checks"] == 0),
    ("manifest_readback_terminal_decision", manifest_readback["terminal_decision"] == terminal_decision),
    ("manifest_readback_authorizes_7b1_only", manifest_readback["next_authorized_cell"]["cell_id"] == "7B1"),
    ("manifest_sidecar_valid", sidecar_is_valid(OUTPUTS["manifest"])),
])

failed_readback = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_readback:
    raise RuntimeError(
        "Cell 7B0 failed during readback verification. Failed checks:\n- "
        + "\n- ".join(failed_readback)
    )

# Final immutability verification.
final_upstream_hashes = {
    label: sha256_file(path)
    for label, path in CELL_7A3_PATHS.items()
}
if final_upstream_hashes != dict(observed_upstream_hashes):
    raise AssertionError("One or more frozen Cell 7A3 artifacts changed during Cell 7B0.")


# --------------------------------------------------------------------------------------------------
# 9. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

total_checks = len(prewrite_checks) + len(readback_checks)
passed_checks = total_checks

separator = "=" * 144
print("\n" + separator)
print("EXPERIMENT 2 — STAGE 7B — CELL 7B0")
print("DOWNSTREAM GES-AWARE RAG PROTOCOL AND AUTHORIZATION FREEZE")
print(separator)
print(f"Notebook                                      : {NOTEBOOK_NAME}")
print(f"Project root                                  : {ROOT}")

print("\nUPSTREAM REVERIFICATION")
print(f"Cell 7A3 manifest SHA-256                     : {sha256_file(CELL_7A3_MANIFEST)}")
print(f"Cell 7A3 score-table SHA-256                  : {sha256_file(CELL_7A3_SCORE_TABLE)}")
print("Cell 7A3 terminal PASS verified               : YES")
print(f"Cell 7A3 manifest QC                          : {manifest_qc_passed}/{manifest_qc_total} PASS")
print(f"Cell 7A3 QC record                            : {record_qc_passed}/{record_qc_total} PASS")
print(f"Cell 7A3 score rows                           : {observed_rows:,}")
print(f"Cell 7A3 score columns                        : {observed_columns}")

print("\nFROZEN EXPERIMENTAL DESIGN")
print(f"Experimental conditions                      : {len(conditions)}")
print("Primary intervention                         : Full-GES soft reranking")
print("Primary comparison                           : D vs A")
print("Mandatory comparators                        : Review/conflict, combined metadata,")
print("                                                 no-star GES, random-quality control")
print("Semantic candidate pool                      : top-20, identical across conditions")
print("Final context                                : top-5")
print("Soft rank fusion                             : 0.75 semantic + 0.25 quality RRF")
print("Hard evidence exclusion                      : PROHIBITED")
print("Scores exposed to LLM                        : NO")
print("Target question set                          : 80 score-blind questions")
print("Primary endpoint                             : citation-supported factual accuracy")
print("Paired bootstrap                             : 2,000 question-level replicates")

print("\nCELL 7B0 FROZEN OUTPUTS")
for label, key in [
    ("Downstream RAG protocol", "protocol"),
    ("Experimental condition inventory", "condition_inventory"),
    ("Evaluation metric inventory", "metric_inventory"),
    ("Leakage/invariance control inventory", "control_inventory"),
    ("QC record", "qc"),
    ("Manifest", "manifest"),
]:
    path = OUTPUTS[key]
    print(f"{label:<46}: {path}")
    print(f"{'SHA-256':<46}: {sha256_file(path)}")

print(f"\nQC checks                                      : {passed_checks}/{total_checks} PASS")

print("\nSCIENTIFIC OPERATIONS")
print("Evidence packets materialized                 : NO")
print("RAG corpus constructed                        : NO")
print("Embeddings generated                          : NO")
print("Retrieval executed                            : NO")
print("Quality reranking executed                    : NO")
print("Questions generated                           : NO")
print("Prompts generated                             : NO")
print("LLM called                                     : NO")
print("Threshold or weight optimization              : NO")
print("Hard evidence exclusion applied               : NO")

print("\nNEXT AUTHORIZED CELL")
print("Cell 7B1                                      : Score-blind evidence-unit, eligibility,")
print("                                                 field-derivability, and question-stratum")
print("                                                 preflight only")
print("Cell 7A3 score loading                        : PROHIBITED")
print("Evidence-packet materialization               : PROHIBITED")
print("Corpus / embeddings / retrieval               : PROHIBITED")
print("Question / prompt / LLM generation            : PROHIBITED")

print(f"\nFINAL DECISION                                : {terminal_decision}")
print(separator)


Mounted at /content/drive

EXPERIMENT 2 — STAGE 7B — CELL 7B0
DOWNSTREAM GES-AWARE RAG PROTOCOL AND AUTHORIZATION FREEZE
Notebook                                      : 05_GES_Aware_Genomic_RAG_Cell_7B0_V2.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM REVERIFICATION
Cell 7A3 manifest SHA-256                     : 99bcff934f5e0c15d8357450fb3992eccfeb972ab61ddba8431a843c13b532dd
Cell 7A3 score-table SHA-256                  : e9b162c9add5aed34a4d68d8bf625251293a18c549945468e627a498843eb802
Cell 7A3 terminal PASS verified               : YES
Cell 7A3 manifest QC                          : 50/50 PASS
Cell 7A3 QC record                            : 50/50 PASS
Cell 7A3 score rows                           : 100,920
Cell 7A3 score columns                        : 28

FROZEN EXPERIMENTAL DESIGN
Experimental conditions                      : 6
Primary intervention                         : Full-GES soft reranking
Primary comparison